# Task 2 — Definição do cenário de previsão

## 1. Introdução e objetivo

O projeto analisa sessões de carregamento de veículos elétricos e pretende prever se uma sessão terminará de forma anormal. A variável `is_abnormal` representa o resultado da sessão: `0` corresponde a um término normal e `1` a um término anormal (regra definida na secção 2).

A previsão é feita no início da sessão, pelo que o classificador só pode usar informação disponível nesse instante. Usar dados recolhidos durante ou após a sessão daria resultados otimistas que não se reproduziriam em utilização real.

Esta Task 2 não treina modelos. Define o alvo, decide que colunas podem ser preditores e fundamenta cada decisão com verificações sobre os dados.

## 2. Carregamento e verificação dos dados

Os nomes das colunas são normalizados como na Task 1: remoção de espaços exteriores, minúsculas, caracteres não alfanuméricos substituídos por `_` e remoção de `_` nas extremidades. As colunas de data são convertidas para séries separadas (`order_dt`, `start_dt`, `end_dt`), sem alterar o dataframe.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("Charging_Data_educational.csv")
REQUIRED_SOURCE_COLUMNS = {
    "end_cause", "order_creation_time", "start_time", "end_time", "payment_time",
    "charging_post_id", "userid", "district_name", "tariff_period", "electricity_price",
    "temperature", "relative_humidity", "precipitation_mm",
}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset não encontrado: {DATA_PATH.resolve()}")
df = pd.read_csv(DATA_PATH)
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
    .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True).str.strip("_")
)
missing_required_columns = REQUIRED_SOURCE_COLUMNS.difference(df.columns)
if missing_required_columns:
    raise KeyError(f"Colunas necessárias ausentes: {sorted(missing_required_columns)}")

order_dt, start_dt, end_dt = (
    pd.to_datetime(df[c], errors="coerce")
    for c in ["order_creation_time", "start_time", "end_time"]
)

print(f"Ficheiro: {DATA_PATH.name}")
print(f"Dimensões antes do alvo: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print("\nTipos de dados após a normalização:")
display(df.dtypes.rename("tipo").to_frame())

### Construção do alvo

`is_abnormal = 1` quando `end_cause` não é `Charging ends normally` nem `User stops charging`. Um `end_cause` ausente fica ausente no alvo, em vez de ser convertido em 1. A tabela de revisão mostra a classe atribuída a cada motivo; os valores em falta são apenas contados, não preenchidos, e nenhuma linha é removida.

In [ ]:
NORMAL_END_CAUSES = {"Charging ends normally", "User stops charging"}
observed_end_causes = set(df["end_cause"].dropna().unique())
missing_end_cause = int(df["end_cause"].isna().sum())

end_cause_review = (
    df["end_cause"].value_counts()
    .rename("observacoes").rename_axis("end_cause").reset_index()
)
end_cause_review["classe"] = np.where(
    end_cause_review["end_cause"].isin(NORMAL_END_CAUSES), 0, 1
)

df["is_abnormal"] = (
    (~df["end_cause"].isin(NORMAL_END_CAUSES)).astype("Int64")
    .mask(df["end_cause"].isna())
)

target_distribution = (
    df["is_abnormal"].value_counts(dropna=False)
    .rename_axis("is_abnormal").reset_index(name="observacoes")
    .sort_values("is_abnormal", na_position="last", ignore_index=True)
)
target_distribution["percentagem"] = (
    target_distribution["observacoes"] / len(df) * 100
).round(3)

missing_summary = df.isna().sum().rename("valores_em_falta").to_frame()
missing_summary["percentagem"] = (missing_summary["valores_em_falta"] / len(df) * 100).round(3)
missing_summary = missing_summary[missing_summary["valores_em_falta"] > 0]

print("Classe atribuída a cada categoria de end_cause:")
display(end_cause_review)
print(f"Valores em falta em end_cause: {missing_end_cause}")
print("\nDistribuição do alvo:")
display(target_distribution)
print("Colunas com valores em falta (não preenchidos):")
display(missing_summary)

### Validação da regra do alvo e do histórico

O dataset traz atributos históricos (`post_*`, `user_*`) que dependem de quais sessões contam como "anormais" e de quando uma sessão conta como "anterior". Em vez de aceitar essa construção por declaração, o código recalcula, numa amostra aleatória fixa, as contagens a partir de `start_time`, `end_time` e `end_cause`, e compara-as com as colunas do dataset.

A construção testada é: sessões do mesmo posto/utilizador cujo `end_time` está dentro da janela anterior ao `start_time` da sessão atual. Isto exclui a sessão atual, sessões futuras e sessões ainda em curso. O recálculo testa também a definição de anormal: apenas as falhas, ou todos os términos diferentes de `Charging ends normally` (alternativa que inclui `User stops charging`).

In [ ]:
HISTORY_SAMPLE_SIZE = 2000
HISTORY_TOLERANCE = 0.995
POST_WINDOWS_DAYS = (1, 7, 30)
USER_WINDOWS_DAYS = (30, 180, 730)

work = df.assign(
    start_dt=start_dt,
    end_dt=end_dt,
    fault=(~df["end_cause"].isin(NORMAL_END_CAUSES)).astype(int),
    not_normal_only=df["end_cause"].ne("Charging ends normally").astype(int),
)
by_post = {key: g for key, g in work.groupby("charging_post_id")}
by_user = {key: g for key, g in work.groupby("userid")}


def completed_in_window(sessions, start, days):
    ended = sessions["end_dt"]
    return sessions[(ended <= start) & (ended > start - pd.Timedelta(days=days))]


def same_gap(expected, observed, tol=0.01):
    if pd.isna(expected) or pd.isna(observed):
        return bool(pd.isna(expected) and pd.isna(observed))
    return abs(expected - observed) < tol


recomputed = []
alt_rule = []
for idx, row in df.sample(HISTORY_SAMPLE_SIZE, random_state=0).iterrows():
    start = start_dt.loc[idx]
    post, user = by_post[row["charging_post_id"]], by_user[row["userid"]]
    result = {}
    for days in POST_WINDOWS_DAYS:
        window = completed_in_window(post, start, days)
        result[f"post_previous_sessions_{days}d"] = (
            len(window) == row[f"post_previous_sessions_{days}d"]
        )
        result[f"post_previous_abnormal_{days}d"] = (
            window["fault"].sum() == row[f"post_previous_abnormal_{days}d"]
        )
        if days == 1:
            alt_rule.append(window["not_normal_only"].sum() == row["post_previous_abnormal_1d"])
    for days in USER_WINDOWS_DAYS:
        window = completed_in_window(user, start, days)
        result[f"user_previous_sessions_{days}d"] = (
            len(window) == row[f"user_previous_sessions_{days}d"]
        )
        result[f"user_previous_abnormal_{days}d"] = (
            window["fault"].sum() == row[f"user_previous_abnormal_{days}d"]
        )
    last_post_end = post.loc[post["end_dt"] <= start, "end_dt"].max()
    last_user_end = user.loc[user["end_dt"] <= start, "end_dt"].max()
    result["post_hours_since_previous_completed_session"] = same_gap(
        (start - last_post_end).total_seconds() / 3600,
        row["post_hours_since_previous_completed_session"],
    )
    result["post_no_previous_completed_session"] = (
        bool(row["post_no_previous_completed_session"]) == bool(pd.isna(last_post_end))
    )
    result["user_days_since_previous_completed_session"] = same_gap(
        (start - last_user_end).total_seconds() / 86400,
        row["user_days_since_previous_completed_session"],
    )
    last_post_fault = post.loc[(post["end_dt"] <= start) & (post["fault"] == 1), "end_dt"].max()
    last_user_fault = user.loc[(user["end_dt"] <= start) & (user["fault"] == 1), "end_dt"].max()
    result["post_hours_since_previous_completed_abnormal"] = same_gap(
        (start - last_post_fault).total_seconds() / 3600,
        row["post_hours_since_previous_completed_abnormal"],
    )
    result["post_no_previous_completed_abnormal"] = (
        bool(row["post_no_previous_completed_abnormal"]) == bool(pd.isna(last_post_fault))
    )
    result["user_days_since_previous_completed_abnormal"] = same_gap(
        (start - last_user_fault).total_seconds() / 86400,
        row["user_days_since_previous_completed_abnormal"],
    )
    result["user_no_previous_completed_abnormal"] = (
        bool(row["user_no_previous_completed_abnormal"]) == bool(pd.isna(last_user_fault))
    )
    result["user_active_sessions_at_start"] = (
        ((user["start_dt"] < start) & (user["end_dt"] > start)).sum()
        == row["user_active_sessions_at_start"]
    )
    result["is_new_user"] = bool(user["start_dt"].lt(start).sum() == 0) == bool(row["is_new_user"])
    recomputed.append(result)

history_match = pd.DataFrame(recomputed).mean().rename("taxa_de_coincidencia")
alt_rule_match = float(np.mean(alt_rule))

print(f"Amostra: {HISTORY_SAMPLE_SIZE:,} sessões (semente fixa)")
display(history_match.to_frame().style.format("{:.2%}"))
print(f"Definição alternativa de anormal (tudo exceto 'Charging ends normally'): "
      f"coincidência em post_previous_abnormal_1d = {alt_rule_match:.2%}")

Se a coincidência for praticamente total, as contagens do dataset usam sessões já terminadas antes do início da sessão atual, e a definição de anormal do dataset é a mesma do alvo (só falhas); a alternativa que inclui `User stops charging` deve coincidir muito menos. Os valores estão nas tabelas acima e são testados na secção 5.

### Disponibilidade temporal de `order_creation_time`

A disponibilidade de `order_creation_time` não é assumida por a coluna estar preenchida. A tabela compara-a com `start_time` e mostra a dimensão da diferença, em minutos (valores negativos: encomenda criada antes do início).

In [ ]:
valid_both = order_dt.notna() & start_dt.notna()
order_minus_start_min = ((order_dt - start_dt)[valid_both].dt.total_seconds() / 60)

order_comparison = pd.DataFrame({
    "relacao": ["Anterior", "Igual", "Posterior"],
    "observacoes": [
        int((order_minus_start_min < 0).sum()),
        int((order_minus_start_min == 0).sum()),
        int((order_minus_start_min > 0).sum()),
    ],
})
order_comparison["percentagem_observacoes_validas"] = (
    order_comparison["observacoes"] / len(order_minus_start_min) * 100
).round(3)

order_gap_quantiles = (
    order_minus_start_min
    .quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1])
    .rename("minutos (order_creation_time - start_time)").to_frame()
)
share_within_5min = float((order_minus_start_min.abs() <= 5).mean())

print(f"Observações com ambas as datas válidas: {valid_both.sum():,} de {len(df):,}")
display(order_comparison)
display(order_gap_quantiles)
print(f"Diferença absoluta até 5 minutos: {share_within_5min:.2%} das observações válidas")

Na maioria das sessões, `order_creation_time` é posterior a `start_time`, e a diferença típica é de poucos minutos. A coluna não está garantidamente disponível no instante da previsão e acrescenta pouco a `start_time`; a decisão é **excluir** (ver secção 4).

### Tarifa, preço e meteorologia

`tariff_period` e `electricity_price` seriam elegíveis se fossem determinados por `start_time`. As variáveis meteorológicas são elegíveis se forem uma série externa associada ao distrito e à hora, e não uma medição da própria sessão. O código testa estas condições nos dados.

In [ ]:
WEATHER_COLUMNS = ["temperature", "relative_humidity", "precipitation_mm"]
start_hour = start_dt.dt.hour

tariff_hours = start_hour.groupby(df["tariff_period"]).agg(lambda h: sorted(h.unique()))
tariffs_per_hour = df.groupby(start_hour)["tariff_period"].nunique()
price_per_tariff = df.groupby("tariff_period")["electricity_price"].agg(["nunique", "min", "max"])

values_per_district_hour = (
    df.groupby([df["district_name"], start_dt.dt.floor("h")])[WEATHER_COLUMNS].nunique()
)
weather_constant_share = (values_per_district_hour <= 1).mean().rename("grupos_com_valor_unico")

print("Horas de início associadas a cada período tarifário:")
display(tariff_hours.rename("horas").to_frame())
print(f"Máximo de períodos tarifários numa mesma hora de início: {tariffs_per_hour.max()}")
print("Preços por período tarifário:")
display(price_per_tariff)
print(f"Grupos (distrito, hora de início): {len(values_per_district_hour):,}")
print("Proporção de grupos em que a variável meteorológica tem um único valor:")
display(weather_constant_share.to_frame().style.format("{:.2%}"))

O período tarifário é função da hora de início e o preço é função do período, pelo que ambos são conhecidos no início da sessão. Os valores meteorológicos são iguais para todas as sessões do mesmo distrito e hora, o que indica uma série externa por hora. Os dados não permitem saber se esse valor é uma medição ou uma previsão, nem se a hora completa já estava disponível no instante do início; esta limitação fica registada na secção 5.

## 3. Formulação do problema de classificação

- **Unidade de observação:** uma sessão de carregamento de um veículo elétrico.
- **Variável-alvo:** `is_abnormal`.
- **Tipo de problema:** classificação binária.
- **Classes:** `0` — término normal; `1` — término anormal (falha), segundo `end_cause`.
- **Momento da previsão:** início da sessão de carregamento.
- **Informação admissível:** informação conhecida nesse instante, incluindo contexto e histórico calculado sem usar a sessão atual nem sessões futuras.
- **Informação não admissível:** informação observada durante ou depois da sessão, dependente da sua conclusão, ou que revele diretamente o resultado.

O classificador recebe a descrição disponível no início de uma sessão e atribui-a à classe normal ou anormal. A decisão não pode depender do que aconteceu depois do instante em que a previsão teria de ser produzida.

## 4. Análise e seleção dos preditores

A tabela documenta individualmente todas as colunas. Cada decisão baseia-se no nome da coluna, no tipo, nos valores observados, na Task 1 e nas verificações da secção 2. Como o critério de disponibilidade é o mesmo para todas, as colunas cuja disponibilidade foi confirmada nos dados são elegíveis e as restantes são excluídas com o motivo indicado.

Tratamento dos identificadores, alinhado com a Task 1: `userid` é excluído porque é quase único por utilizador e favorece a memorização de padrões individuais (o histórico `user_*` já resume a informação útil); `charging_post_id` mantém-se como categórica, com risco de memorização a controlar na validação e uma versão de comparação sem o identificador.

In [ ]:
VALID_DECISIONS = {
    "Elegível",
    "Excluir — variável-alvo",
    "Excluir — construção do alvo",
    "Excluir — fuga de informação",
    "Excluir — disponibilidade não garantida",
    "Excluir — risco de memorização",
    "Pendente de confirmação",
}
VALID_AVAILABILITY = {"Sim", "Não", "Por confirmar"}

HISTORY_JUSTIFICATION = (
    "Conta apenas sessões já terminadas antes do início da sessão atual, sem a sessão atual, "
    "sem sessões futuras e sem sessões ainda em curso (recalculado em amostra na secção 2). "
    "Os indicadores de ausência de histórico tratam o cold-start. Mantêm-se riscos de "
    "privacidade, memorização e fraca generalização para novos utilizadores ou postos."
)
WEATHER_JUSTIFICATION = (
    "Valor igual para todas as sessões do mesmo distrito e hora de início (secção 2): é uma "
    "série externa por hora, não uma medição da sessão. Não se sabe se é medição ou previsão."
)


def make_decision(decision, availability, category, meaning, justification):
    if decision not in VALID_DECISIONS:
        raise ValueError(f"Decisão inválida: {decision}")
    if availability not in VALID_AVAILABILITY:
        raise ValueError(f"Disponibilidade inválida: {availability}")
    return {
        "significado": meaning,
        "tipo_categoria": category,
        "disponibilidade_no_inicio": availability,
        "decisao": decision,
        "justificacao": justification,
    }


metadata = {
    "userid": make_decision(
        "Excluir — risco de memorização", "Sim", "Identificador", "Identificador do utilizador",
        "Disponível no início, mas excluído do primeiro modelo (Task 1): identificador quase "
        "único que favorece a memorização de padrões individuais e generaliza mal para novos "
        "utilizadores. O histórico user_* cobre a informação útil.",
    ),
    "charging_post_id": make_decision(
        "Elegível", "Sim", "Identificador", "Identificador do posto de carregamento",
        "Posto escolhido no início da sessão. Mantém-se como categórica para modelos de "
        "árvore (Task 1); o risco de memorização é controlado com uma versão de comparação "
        "sem o identificador e com validação temporal.",
    ),
    "location_information": make_decision(
        "Elegível", "Sim", "Localização", "Localização ou tipo de local associado à sessão",
        "Contexto do posto, conhecido no início, embora possa substituir características "
        "não observadas.",
    ),
    "district_name": make_decision(
        "Elegível", "Sim", "Localização", "Distrito associado à sessão",
        "Contexto geográfico do posto, conhecido no início da sessão.",
    ),
    "order_creation_time": make_decision(
        "Excluir — disponibilidade não garantida", "Não", "Data/hora",
        "Data e hora de criação da encomenda",
        "Posterior a start_time na maioria das sessões e com diferença de poucos minutos "
        "(secção 2): não está garantida no instante da previsão e acrescenta pouco a start_time.",
    ),
    "start_time": make_decision(
        "Elegível", "Sim", "Data/hora", "Data e hora de início da sessão",
        "Instante de referência da previsão. Entra apenas através de atributos derivados "
        "(hora, dia da semana, mês, fim de semana), nunca como timestamp bruto.",
    ),
    "end_time": make_decision(
        "Excluir — fuga de informação", "Não", "Data/hora", "Data e hora de fim da sessão",
        "Só é conhecida depois da sessão e revela informação posterior ao instante da previsão.",
    ),
    "payment_time": make_decision(
        "Excluir — fuga de informação", "Não", "Data/hora", "Data e hora do pagamento",
        "Registada durante ou depois da sessão; não está disponível no início.",
    ),
    "transaction_power_kwh": make_decision(
        "Excluir — fuga de informação", "Não", "Energia", "Energia transacionada/consumida",
        "O nome indica energia (kWh), um acumulado da sessão só conhecido no fim.",
    ),
    "electricity_cost_yuan": make_decision(
        "Excluir — fuga de informação", "Não", "Custo", "Custo de eletricidade",
        "Calculado a partir da energia transacionada, só conhecida no fim da sessão.",
    ),
    "service_charge_yuan": make_decision(
        "Excluir — fuga de informação", "Não", "Custo", "Taxa de serviço",
        "Montante da transação, não documentado como conhecido antes da sessão.",
    ),
    "transaction_amount_yuan": make_decision(
        "Excluir — fuga de informação", "Não", "Custo", "Montante da transação",
        "Calculado no contexto da transação e dependente da sua conclusão.",
    ),
    "actual_payment_yuan": make_decision(
        "Excluir — fuga de informação", "Não", "Custo", "Montante efetivamente pago",
        "O pagamento é posterior ao início e reflete a sessão concluída.",
    ),
    "end_cause": make_decision(
        "Excluir — construção do alvo", "Não", "Construção do alvo",
        "Motivo registado para o término",
        "Usada para construir is_abnormal; revela diretamente o resultado da sessão.",
    ),
    "temperature": make_decision(
        "Elegível", "Sim", "Condição ambiental", "Temperatura associada ao distrito e hora",
        WEATHER_JUSTIFICATION,
    ),
    "relative_humidity": make_decision(
        "Elegível", "Sim", "Condição ambiental", "Humidade relativa associada ao distrito e hora",
        WEATHER_JUSTIFICATION,
    ),
    "precipitation_mm": make_decision(
        "Elegível", "Sim", "Condição ambiental", "Precipitação associada ao distrito e hora",
        WEATHER_JUSTIFICATION,
    ),
    "tariff_period": make_decision(
        "Elegível", "Sim", "Contexto tarifário", "Período tarifário",
        "Cada hora de start_time pertence a um único período (secção 2): é conhecido no início.",
    ),
    "electricity_price": make_decision(
        "Elegível", "Sim", "Contexto tarifário", "Preço da eletricidade",
        "Cada período tarifário tem um único preço (secção 2); como o período é conhecido no "
        "início, o preço também é. Redundante com tariff_period.",
    ),
    "is_new_user": make_decision(
        "Elegível", "Sim", "Histórico/agregado", "Indicador de utilizador novo",
        "Verdadeiro quando o utilizador não tem sessões anteriores (recalculado na secção 2). "
        + HISTORY_JUSTIFICATION,
    ),
    "user_active_sessions_at_start": make_decision(
        "Elegível", "Sim", "Histórico/agregado", "Sessões ativas do utilizador no início",
        "Sessões do utilizador iniciadas antes e ainda a decorrer no início da sessão atual "
        "(recalculado na secção 2). " + HISTORY_JUSTIFICATION,
    ),
}

history_columns = [
    c for c in df.columns
    if c.startswith(("post_", "user_previous_", "user_days_since_", "user_no_previous_"))
]
for column in history_columns:
    metadata.setdefault(column, make_decision(
        "Elegível", "Sim", "Histórico/agregado", "Atributo histórico do posto ou utilizador",
        HISTORY_JUSTIFICATION,
    ))

rows = []
for column in df.columns:
    if column == "is_abnormal":
        info = make_decision(
            "Excluir — variável-alvo", "Não", "Alvo", "Indicador binário de término anormal",
            "É o resultado que se pretende prever e não pode ser preditor.",
        )
    elif column in metadata:
        info = metadata[column]
    else:
        info = make_decision(
            "Pendente de confirmação", "Por confirmar", "Não determinado",
            "O significado não pode ser confirmado apenas com a informação disponível",
            "É necessário confirmar o significado e a disponibilidade temporal antes de usar "
            "a coluna.",
        )
    rows.append({"nome_coluna": column, **info})

predictor_decisions = pd.DataFrame(rows)
display(
    predictor_decisions.style.set_properties(
        subset=["justificacao"],
        **{"white-space": "normal", "max-width": "500px", "min-width": "350px"},
    )
)

### Resumo das decisões

O resumo é derivado da tabela anterior, pelo que as listas e contagens não são escritas à mão. As listas de preditores e de exclusões são exportadas para `predictors_task2.json`, para serem reutilizadas pelo pipeline da Task 1 e dos modelos seguintes.

In [ ]:
import json

decision_counts = predictor_decisions["decisao"].value_counts().rename("numero_colunas").to_frame()
eligible_predictors = predictor_decisions.loc[
    predictor_decisions["decisao"] == "Elegível", "nome_coluna"
].tolist()
excluded_by_decision = {
    decision: group["nome_coluna"].tolist()
    for decision, group in predictor_decisions.groupby("decisao")
    if decision != "Elegível"
}
pending_columns = excluded_by_decision.get("Pendente de confirmação", [])

print(f"Número total de colunas analisadas: {len(predictor_decisions)}")
display(decision_counts)
print(f"Preditores elegíveis ({len(eligible_predictors)}):")
print(eligible_predictors)
for decision, columns in excluded_by_decision.items():
    print(f"\n{decision} ({len(columns)}):")
    print(columns)

EXPORT_PATH = Path("predictors_task2.json")
EXPORT_PATH.write_text(
    json.dumps(
        {
            "target": "is_abnormal",
            "eligible_predictors": eligible_predictors,
            "excluded_by_decision": excluded_by_decision,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"\nExportado para {EXPORT_PATH.resolve()}")

## 5. Verificação das decisões

Controlo de qualidade das decisões. Cada condição recebe um estado: **Verificado** (satisfeita), **Problema** (inconsistência detetada; o notebook falha) ou **Não verificável** (os dados não permitem confirmar). As verificações assentam em evidência dos dados (alvo, histórico, tarifa, meteorologia, datas) e em restrições explícitas e independentes da tabela de decisões. Não treinam modelos nem substituem a validação temporal.

In [ ]:
# Colunas que, por construção do problema, nunca podem ser preditores.
INADMISSIBLE_PREDICTORS = {
    "is_abnormal", "end_cause", "end_time", "payment_time", "transaction_power_kwh",
    "electricity_cost_yuan", "service_charge_yuan", "transaction_amount_yuan",
    "actual_payment_yuan",
}

normal_rows_are_zero = all(
    (df.loc[df["end_cause"] == cause, "is_abnormal"] == 0).all() for cause in NORMAL_END_CAUSES
)
non_normal_rows_are_one = (
    df.loc[df["end_cause"].notna() & ~df["end_cause"].isin(NORMAL_END_CAUSES), "is_abnormal"] == 1
).all()
missing_end_cause_stays_missing = df.loc[df["end_cause"].isna(), "is_abnormal"].isna().all()
documented = predictor_decisions["nome_coluna"]

checks = [
    ("O alvo só tem as classes 0 e 1 e não tem valores em falta",
     bool(df["is_abnormal"].isin([0, 1]).all()), ""),
    ("As duas categorias normais existem em end_cause",
     NORMAL_END_CAUSES <= observed_end_causes, ""),
    ("Términos normais são 0, anormais são 1 e end_cause ausente fica ausente",
     bool(normal_rows_are_zero and non_normal_rows_are_one and missing_end_cause_stays_missing), ""),
    ("A regra do alvo (só falhas) coincide com o histórico de anormais do dataset",
     bool(history_match.filter(like="abnormal").min() >= HISTORY_TOLERANCE and alt_rule_match < 0.5),
     f"alternativa: {alt_rule_match:.2%}"),
    ("O histórico do dataset coincide com o recálculo cronológico (sessões terminadas antes do início)",
     bool(history_match.min() >= HISTORY_TOLERANCE), f"mínimo: {history_match.min():.2%}"),
    ("order_creation_time é posterior a start_time em parte das sessões",
     bool((order_minus_start_min > 0).any()), f"{(order_minus_start_min > 0).mean():.2%}"),
    ("Cada hora de início tem um só período tarifário e cada período um só preço",
     bool(tariffs_per_hour.max() <= 1 and price_per_tariff["nunique"].max() <= 1), ""),
    ("Meteorologia constante por distrito e hora de início",
     bool(weather_constant_share.min() == 1.0), ""),
    ("A meteorologia é medição ou previsão disponível antes do fim da hora",
     None, "o dataset não documenta a origem dos valores"),
    ("Todas as colunas estão documentadas uma única vez",
     bool(documented.is_unique and set(documented) == set(df.columns)), ""),
    ("Não há colunas pendentes de confirmação", not pending_columns, ""),
    ("Nenhuma variável proibida está entre os preditores elegíveis",
     not (INADMISSIBLE_PREDICTORS & set(eligible_predictors)), ""),
    ("Identificadores tratados como na Task 1 (userid excluído, charging_post_id elegível)",
     "userid" not in eligible_predictors and "charging_post_id" in eligible_predictors, ""),
    ("Todos os atributos históricos são elegíveis",
     set(history_columns) <= set(eligible_predictors), f"{len(history_columns)} colunas"),
]

STATUS = {True: "Verificado", False: "Problema", None: "Não verificável"}
verification = pd.DataFrame(
    [{"verificacao": name, "estado": STATUS[ok], "nota": note} for name, ok, note in checks]
)
display(
    verification.style.set_properties(
        subset=["verificacao", "nota"], **{"white-space": "normal", "text-align": "left"}
    )
)
if (verification["estado"] == "Problema").any():
    raise AssertionError("Há verificações com estado Problema; rever as decisões acima.")

A única verificação não verificável é a origem dos valores meteorológicos: os dados não distinguem medição de previsão. Se for medição da hora completa, estaria apenas disponível no fim da hora; se for uma previsão, estaria disponível no início. Este risco deve ser confirmado com a fonte dos dados, ou testado com a meteorologia da hora anterior.

## 6. Objetivos de avaliação e consequências dos erros

Considerando `is_abnormal = 1` como a classe positiva, o classificador produz uma decisão no início da sessão que pode desencadear ações operacionais.

- **Falso positivo (FP):** sessão normal sinalizada como anormal. Pode gerar monitorização adicional, inspeção ou manutenção desnecessária e intervenção junto do utilizador; consome recursos e causa fadiga de alertas.
- **Falso negativo (FN):** sessão anormal classificada como normal. Uma falha potencial pode não ser monitorizada nem encaminhada para manutenção, perdendo-se a oportunidade de intervir durante a sessão ou de priorizar um posto.

Neste cenário, o **falso negativo tende a ser o erro mais grave**, porque deixa passar uma sessão anormal sem sinalização, enquanto o falso positivo representa sobretudo consumo de recursos. Esta prioridade assume que a finalidade principal do sistema é detetar sessões anormais para ação de acompanhamento.

### Métricas e limiar

- **Métrica principal (independente do limiar):** PR-AUC (`average_precision`), que torna visível o desequilíbrio de classes (a referência de um classificador sem poder discriminativo é a prevalência da classe anormal, não 0,5).
- **Métrica no ponto de operação:** *recall* da classe anormal (fração de sessões anormais detetadas), lida em conjunto com a precisão, para que o número de falsos positivos fique visível.
- **Ponto de operação:** *recall* a uma precisão mínima fixada com a operação, ou $F_\beta$ com $\beta > 1$ se o *recall* for priorizado. A matriz de confusão deve ser reportada no limiar escolhido.
- **Limiar:** com probabilidades bem calibradas, o limiar que minimiza o custo esperado é $t = c_{FP} / (c_{FP} + c_{FN})$. O quadro abaixo mostra como o limiar desce quando o falso negativo é mais caro. As razões de custo são hipóteses ilustrativas, a validar com a operação.

A avaliação usa folds temporais com janela de treino crescente (secção 7), e não validação aleatória, para refletir a utilização real.

In [ ]:
prevalence = float(df["is_abnormal"].mean())
reference_values = pd.DataFrame({
    "referencia": [
        "Prevalência da classe anormal",
        "PR-AUC de um classificador sem poder discriminativo",
        "Precisão ao sinalizar todas as sessões como anormais",
        "Recall ao não sinalizar nenhuma sessão",
    ],
    "valor": [prevalence, prevalence, prevalence, 0.0],
})
cost_ratios = [1, 2, 5, 10, 20]
threshold_table = pd.DataFrame({"custo_FN_sobre_custo_FP": cost_ratios})
threshold_table["limiar_otimo"] = (1 / (1 + threshold_table["custo_FN_sobre_custo_FP"])).round(3)

display(reference_values.style.format({"valor": "{:.2%}"}))
display(threshold_table)

O problema é uma classificação binária ao nível da sessão, com previsão no início do carregamento, e a prevalência da classe anormal (tabela acima) mostra que os dados são desequilibrados; as métricas devem ser lidas à luz desse desequilíbrio.

A elegibilidade exige informação disponível no início e independente da conclusão da sessão. O alvo foi validado contra o histórico do dataset (só falhas contam como anormais, não `User stops charging`) e os atributos históricos foram recalculados cronologicamente a partir das sessões terminadas antes do início. Tarifa e preço são determinados por `start_time` e são elegíveis; a meteorologia é uma série externa por distrito e hora, elegível com a limitação de a sua origem não ser verificável. `order_creation_time` é excluída por ser posterior ao início na maioria das sessões, e `userid` por risco de memorização.

As variáveis de término, pagamento, energia e custos finais, bem como `end_cause`, são excluídas por dependerem de informação posterior ou por construírem o alvo. A lista final de preditores elegíveis está em `predictors_task2.json`.

Os falsos negativos tendem a ser mais graves, e a análise do limiar mostra que a razão de custos define o ponto de operação. As métricas e o limiar finais serão definidos na fase de modelação, com a razão de custos validada.

## 7. Métricas, estabilidade temporal e baseline

In [22]:
# Distribuição das classes
class_counts = df["is_abnormal"].value_counts().sort_index()

# Percentagem de cada classe
class_percentages = (
    df["is_abnormal"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print("Número de sessões por classe:")
print(class_counts)

print("\nPercentagem de sessões por classe:")
print(class_percentages.round(2))

Número de sessões por classe:
is_abnormal
0    359143
1     81934
Name: count, dtype: Int64

Percentagem de sessões por classe:
is_abnormal
0    81.42
1    18.58
Name: proportion, dtype: Float64


A variável-alvo apresenta duas classes: 359 143 sessões normais (81,42%) e 81 934 sessões anormais (18,58%). Embora o desequilíbrio entre classes não seja extremo, a classe anormal é minoritária, pelo que a accuracy, por si só, não permite avaliar adequadamente a capacidade de deteção de anomalias.

Assim, propõe-se utilizar a **PR-AUC** (`average_precision`), como métrica principal. Esta métrica permite avaliar a capacidade do modelo para identificar sessões anormais, considerando o compromisso entre a proporção de anomalias detetadas e a proporção de alertas que correspondem efetivamente a anomalias.

Como métricas complementares, serão utilizadas o recall, a precision, o F2-score, a ROC-AUC, o Matthews Correlation Coefficient (MCC) e o Brier score. Serão também contabilizados os falsos positivos e os falsos negativos por cada 1000 sessões, permitindo interpretar os resultados em termos operacionais. Nos folds temporais, a estabilidade será resumida por média, desvio-padrão e pior fold. O pior fold é o menor valor quando maior é melhor e o maior valor para o Brier score.

A escolha final da métrica principal será acompanhada pela comparação com baselines simples e pela avaliação da estabilidade dos resultados ao longo de diferentes períodos temporais. As baselines são a classe maioritária e um score aleatório proporcional à prevalência. Para um score aleatório, espera-se PR-AUC aproximadamente igual à prevalência e ROC-AUC próximo de 0,5.

### 7.1. Função para calcular as métricas
A função recebe as classes reais (y_true) e uma pontuação/probabilidade para a classe anormal (y_score). O limiar converte as pontuações em classes previstas para as métricas que dependem de uma decisão binária.

- AP (PR-AUC): métrica principal, calculada a partir das pontuações.
- Recall: fração das sessões anormais detetadas.
- Precision: fração dos alertas que são realmente anormais.
- F2: dá mais peso ao recall do que à precision.
- ROC-AUC: capacidade de ordenar sessões anormais acima das normais.
- MCC: métrica baseada nos quatro resultados da matriz de confusão.
- Brier score: erro das probabilidades; valores mais baixos são melhores.
- FP/FN por 1000: tradução dos erros para uma escala operacional.

O limiar 0.5 é apenas o valor por defeito para exemplos. Não deve ser considerado automaticamente o limiar ótimo.

In [23]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score,
    roc_auc_score,
    matthews_corrcoef,
    brier_score_loss,
    confusion_matrix,
)

def calculate_metrics(y_true, y_score, threshold=0.5):
    """Calcula métricas de classificação binária para a classe positiva 1."""
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)

    # Validações de entrada
    if y_true.ndim != 1 or y_score.ndim != 1:
        raise ValueError("y_true e y_score devem ser vetores unidimensionais.")
    if len(y_true) == 0:
        raise ValueError("y_true não pode estar vazio.")
    if len(y_true) != len(y_score):
        raise ValueError("y_true e y_score têm de ter o mesmo comprimento.")
    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("y_true deve conter apenas 0 e 1.")
    if not np.isfinite(y_score).all():
        raise ValueError("y_score contém valores não finitos.")
    if ((y_score < 0) | (y_score > 1)).any():
        raise ValueError("y_score deve conter probabilidades entre 0 e 1.")
    if not np.isscalar(threshold) or not np.isfinite(threshold):
        raise ValueError("threshold deve ser um valor numérico finito.")
    if not 0 <= threshold <= 1:
        raise ValueError("threshold deve estar entre 0 e 1.")

    # Garantir que as classes são inteiras
    y_true = y_true.astype(int)

    # Converte as probabilidades em previsões binárias com base no limiar fornecido
    y_pred = (y_score >= threshold).astype(int)

    # Calcula a matriz de confusão e extrai os valores de TN, FP, FN e TP
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    # ROC-AUC não está definida se y_true tiver apenas uma classe.
    roc_auc = (
        roc_auc_score(y_true, y_score)
        if len(np.unique(y_true)) == 2 else np.nan
    )

    n = len(y_true)
    return {
        "AP (PR-AUC)": average_precision_score(y_true, y_score),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "F2-score": fbeta_score(
            y_true, y_pred, beta=2, zero_division=0
        ),
        "ROC-AUC": roc_auc,
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Brier score": brier_score_loss(y_true, y_score),
        "FP por 1000 sessões": fp / n * 1000,
        "FN por 1000 sessões": fn / n * 1000,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

A função calculate_metrics definida nesta secção será utilizada nas secções seguintes para avaliar as diferentes abordagens de classificação. A sua definição permite calcular as mesmas métricas de forma consistente, facilitando a comparação entre as baselines e os modelos desenvolvidos.

Nesta fase, a função é apenas definida, pelo que ainda não são apresentados resultados de avaliação. As métricas serão calculadas posteriormente, utilizando as previsões de cada abordagem e os valores reais da classe is_abnormal correspondentes aos conjuntos de teste de cada fold temporal.

### 7.2. Criar folds temporais
Para avaliar as diferentes abordagens de classificação, é necessário dividir os dados em conjuntos de treino e teste. Como as sessões de carregamento ocorrem ao longo do tempo, esta divisão deve respeitar a ordem cronológica das observações.

Neste trabalho, utiliza-se o método TimeSeriesSplit, da biblioteca Scikit-learn, para criar cinco folds temporais. Em cada fold, o conjunto de treino contém observações anteriores às do conjunto de teste, sendo utilizada uma janela de treino crescente (expanding window).

Esta estratégia permite simular um cenário de utilização real, no qual as abordagens são desenvolvidas com base em dados históricos e avaliadas em sessões posteriores. Permite também analisar a estabilidade dos resultados ao longo do tempo e comparar as diferentes abordagens nos mesmos períodos.

Antes de criar os folds, é necessário preparar o DataFrame que contém as observações a utilizar na avaliação.

#### Preparação do conjunto de dados de avaliação

O DataFrame evaluation_df é criado a partir do DataFrame original df e contém as observações que serão utilizadas na avaliação das diferentes abordagens de classificação. A sua criação permite preparar os dados especificamente para a divisão temporal, sem alterar diretamente o DataFrame original.

Inicialmente, é criada uma cópia de df. Em seguida, a coluna start_time é convertida para o formato de data/hora, são removidas as observações sem data de início ou sem valor na variável-alvo is_abnormal e os dados são ordenados cronologicamente. Por fim, os índices são reorganizados para facilitar a seleção das observações em cada fold.

Esta preparação é necessária porque o método TimeSeriesSplit divide os dados de acordo com a ordem das observações. Assim, a ordenação cronológica garante que os conjuntos de treino correspondem a períodos anteriores aos respetivos conjuntos de teste.

O evaluation_df será utilizado nas secções seguintes para selecionar as observações de treino e teste de cada fold, através dos índices armazenados em temporal_folds. Desta forma, as diferentes baselines e os modelos serão avaliados nas mesmas divisões temporais, permitindo uma comparação consistente dos resultados.

In [24]:
# Preparar os dados para a avaliação temporal
evaluation_df = df.copy()

# Garantir que start_time está no formato datetime
evaluation_df["start_time"] = pd.to_datetime(
    evaluation_df["start_time"],
    errors="coerce"
)

# Remover observações sem data de início ou sem classe-alvo
evaluation_df = evaluation_df.dropna(
    subset=["start_time", "is_abnormal"]
)

# Ordenar cronologicamente
evaluation_df = (
    evaluation_df
    .sort_values("start_time")
    .reset_index(drop=True)
)

print("Número de observações:", len(evaluation_df))
print("Data inicial:", evaluation_df["start_time"].min())
print("Data final:", evaluation_df["start_time"].max())
print("Valores da variável-alvo:")
print(evaluation_df["is_abnormal"].value_counts())

Número de observações: 441077
Data inicial: 2019-12-31 21:27:03
Data final: 2021-12-31 23:52:14
Valores da variável-alvo:
is_abnormal
0    359143
1     81934
Name: count, dtype: Int64


Após a preparação dos dados, foram mantidas 441 077 sessões de carregamento, abrangendo o período entre 31 de dezembro de 2019 e 31 de dezembro de 2021.

Relativamente à distribuição da variável-alvo is_abnormal, verificou-se que 359 143 sessões (aproximadamente 81,42%) são classificadas como normais (0), enquanto 81 934 sessões (cerca de 18,58%) são classificadas como anómalas (1).

Estes resultados evidenciam um desequilíbrio entre as duas classes, uma vez que as sessões normais são significativamente mais frequentes do que as anómalas. Este aspeto deve ser considerado na avaliação das abordagens de classificação, pois um modelo que preveja maioritariamente a classe normal poderá apresentar uma elevada exatidão (accuracy) sem conseguir identificar adequadamente as sessões anómalas. Por este motivo, serão utilizadas métricas como a Average Precision (AP), o recall e o F2-score, que permitem avaliar melhor a capacidade de deteção da classe minoritária.

In [25]:
from sklearn.model_selection import TimeSeriesSplit

N_SPLITS = 5
if len(evaluation_df) <= N_SPLITS:
    raise ValueError("Não existem observações suficientes para criar os folds temporais.")

time_splitter = TimeSeriesSplit(n_splits=N_SPLITS)
temporal_folds = list(time_splitter.split(evaluation_df))

for fold_number, (train_idx, test_idx) in enumerate(temporal_folds, start=1):
    train_dates = evaluation_df.iloc[train_idx]["start_time"]
    test_dates = evaluation_df.iloc[test_idx]["start_time"]
    print(
        f"Fold {fold_number}: "
        f"treino até {train_dates.max()} | "
        f"teste de {test_dates.min()} a {test_dates.max()} | "
        f"{len(train_idx):,} treino / {len(test_idx):,} teste"
    )

Fold 1: treino até 2020-07-23 14:56:56 | teste de 2020-07-23 14:57:05 a 2020-11-02 21:43:39 | 73,517 treino / 73,512 teste
Fold 2: treino até 2020-11-02 21:43:39 | teste de 2020-11-02 21:44:16 a 2021-03-17 16:07:35 | 147,029 treino / 73,512 teste
Fold 3: treino até 2021-03-17 16:07:35 | teste de 2021-03-17 16:07:54 a 2021-07-12 14:29:41 | 220,541 treino / 73,512 teste
Fold 4: treino até 2021-07-12 14:29:41 | teste de 2021-07-12 14:30:46 a 2021-10-07 15:20:57 | 294,053 treino / 73,512 teste
Fold 5: treino até 2021-10-07 15:20:57 | teste de 2021-10-07 15:21:14 a 2021-12-31 23:52:14 | 367,565 treino / 73,512 teste


Os resultados mostram que foram criados cinco folds temporais, respeitando a ordem cronológica das sessões de carregamento. Em cada fold, o conjunto de teste contém 73 512 observações, enquanto o conjunto de treino aumenta progressivamente, passando de 73 517 observações no primeiro fold para 367 565 no quinto.

Esta estratégia de janela de treino crescente permite avaliar as diferentes abordagens em períodos temporais distintos, utilizando progressivamente mais dados históricos. Os índices dos conjuntos de treino e teste são armazenados na lista temporal_folds e serão reutilizados nas secções seguintes, garantindo que as baselines e os modelos são avaliados nas mesmas divisões temporais.

Desta forma, será possível comparar o desempenho das diferentes abordagens em condições consistentes e analisar a sua estabilidade ao longo do tempo.

### 7.3. Baseline 1 — classe maioritária
A primeira baseline consiste em prever sempre a classe mais frequente no conjunto de treino de cada fold. No conjunto de dados atual, a classe normal (0) é a maioritária, representando aproximadamente 81,42% das sessões.

O objetivo desta baseline é estabelecer uma referência simples para avaliar se as abordagens seguintes conseguem melhorar a deteção de sessões anómalas. Como todas as sessões são classificadas como normais, esta estratégia poderá apresentar uma accuracy elevada, mas não conseguirá identificar sessões anómalas, evidenciando as limitações desta métrica em problemas com classes desequilibradas.

Para a métrica Average Precision (AP), são atribuídas pontuações constantes a todas as sessões. Como estas pontuações não permitem distinguir as sessões normais das anómalas, a AP corresponde à prevalência da classe positiva no conjunto de teste.

Esta baseline permite, assim, estabelecer um ponto de comparação para as restantes abordagens e demonstrar a importância de utilizar métricas que avaliem especificamente a capacidade de deteção da classe anómala.

In [26]:
import numpy as np

majority_fold_results = []

for fold_number, (train_idx, test_idx) in enumerate(temporal_folds, start=1):
    y_train = evaluation_df.iloc[train_idx]["is_abnormal"].to_numpy()
    y_test = evaluation_df.iloc[test_idx]["is_abnormal"].to_numpy()

    majority_class = int(pd.Series(y_train).mode().iloc[0])
    # Pontuação constante: 0 se a classe maioritária for normal, 1 se for anormal.
    y_score = np.full(len(y_test), float(majority_class))

    result = calculate_metrics(y_test, y_score, threshold=0.5)
    result["fold"] = fold_number
    result["baseline"] = "Classe maioritária"
    majority_fold_results.append(result)

majority_results_df = pd.DataFrame(majority_fold_results)
display(majority_results_df)

,AP (PR-AUC),Recall,Precision,F2-score,ROC-AUC,MCC,Brier score,FP por 1000 sessões,FN por 1000 sessões,TP,TN,FP,FN,fold,baseline
0,0.187371,0.0,0.0,0.0,0.5,0.0,0.187371,0.0,187.370769,0,59738,0,13774,1,Classe maioritária
1,0.164000,0.0,0.0,0.0,0.5,0.0,0.164000,0.0,164.000435,0,61456,0,12056,2,Classe maioritária
2,0.179495,0.0,0.0,0.0,0.5,0.0,0.179495,0.0,179.494504,0,60317,0,13195,3,Classe maioritária
3,0.214441,0.0,0.0,0.0,0.5,0.0,0.214441,0.0,214.441180,0,57748,0,15764,4,Classe maioritária
4,0.171795,0.0,0.0,0.0,0.5,0.0,0.171795,0.0,171.795081,0,60883,0,12629,5,Classe maioritária


Os resultados obtidos são consistentes com o comportamento esperado da baseline da classe maioritária. Em todos os folds, a classe normal (0) é a mais frequente no conjunto de treino, pelo que todas as sessões do conjunto de teste são classificadas como normais.

O recall e o F2-score apresentam valores iguais a zero, uma vez que nenhuma sessão anómala é identificada. Consequentemente, a precision também é zero, de acordo com a convenção utilizada pelo Scikit-learn quando não existem previsões positivas. O MCC é igualmente zero, enquanto o ROC-AUC é 0,5, indicando que as pontuações constantes não permitem distinguir as sessões normais das anómalas.

A Average Precision (AP) varia entre aproximadamente 0,164 e 0,214, correspondendo à prevalência da classe anómala em cada conjunto de teste. O Brier score apresenta os mesmos valores, uma vez que a probabilidade prevista para a classe anómala é zero em todas as sessões. Relativamente às taxas de erro, a baseline apresenta zero falsos positivos por 1000 sessões, mas falha todas as sessões anómalas, originando aproximadamente 164 a 214 falsos negativos por 1000 sessões, consoante o fold.

Conclui-se que esta baseline não é adequada para a deteção de sessões anómalas, apesar de poder apresentar uma accuracy relativamente elevada devido à predominância da classe normal. A sua utilidade consiste em estabelecer uma referência simples, permitindo verificar se as abordagens seguintes conseguem identificar anomalias e melhorar o desempenho relativamente a uma estratégia que prevê sempre a classe maioritária.

### 7.4. Baseline 2 — aleatório proporcional à prevalência
Esta baseline é utilizada como uma segunda referência para avaliar o desempenho dos modelos de classificação desenvolvidos posteriormente. Ao contrário da baseline da classe maioritária, que prevê sempre a classe normal, esta abordagem atribui pontuações aleatórias às sessões, sem utilizar as suas características para identificar anomalias.

O limiar de classificação é definido com base na prevalência da classe anómala no conjunto de treino. Como as pontuações são geradas uniformemente entre 0 e 1, utiliza-se o limiar (1-p), em que (p) representa a proporção de sessões anómalas no treino. Desta forma, espera-se que a proporção de sessões sinalizadas como anómalas seja aproximadamente igual à prevalência observada no treino.

Esta baseline permite estabelecer uma referência para o desempenho de uma abordagem sem capacidade discriminativa. A Average Precision (AP) esperada é aproximadamente igual à prevalência da classe positiva, embora os resultados de cada fold possam variar devido à aleatoriedade.

A comparação com esta baseline permite avaliar se os modelos seguintes conseguem identificar sessões anómalas de forma mais eficaz do que uma abordagem aleatória. A utilização de uma semente aleatória fixa garante a reprodutibilidade dos resultados.

In [27]:
RANDOM_STATE = 42
random_fold_results = []

for fold_number, (train_idx, test_idx) in enumerate(temporal_folds, start=1):
    y_train = evaluation_df.iloc[train_idx]["is_abnormal"].to_numpy()
    y_test = evaluation_df.iloc[test_idx]["is_abnormal"].to_numpy()

    prevalence_train = float(y_train.mean())
    rng = np.random.default_rng(RANDOM_STATE + fold_number)
    y_score = rng.uniform(0, 1, size=len(y_test))
    random_threshold = 1 - prevalence_train

    result = calculate_metrics(
        y_test, y_score, threshold=random_threshold
    )
    result["fold"] = fold_number
    result["baseline"] = "Aleatório proporcional à prevalência"
    result["prevalência_treino"] = prevalence_train
    result["limiar"] = random_threshold
    random_fold_results.append(result)

random_results_df = pd.DataFrame(random_fold_results)
display(random_results_df)

,AP (PR-AUC),Recall,Precision,F2-score,ROC-AUC,MCC,Brier score,FP por 1000 sessões,FN por 1000 sessões,TP,TN,FP,FN,fold,baseline,prevalência_treino,limiar
0,0.189518,0.203717,0.192403,0.201349,0.504421,0.006415,0.332385,160.218740,149.200131,2806,47960,11778,10968,1,Aleatório proporcional à prevalência,0.197451,0.802549
1,0.161984,0.188786,0.161407,0.182591,0.495892,-0.003412,0.334703,160.858091,133.039504,2276,49631,11825,9780,2,Aleatório proporcional à prevalência,0.192411,0.807589
2,0.176534,0.178704,0.177014,0.178363,0.493942,-0.003041,0.334864,149.132114,147.418109,2358,49354,10963,10837,3,Aleatório proporcional à prevalência,0.182941,0.817059
3,0.213668,0.181045,0.214313,0.186846,0.499034,-0.000147,0.333400,142.330504,175.617586,2854,47285,10463,12910,4,Aleatório proporcional à prevalência,0.182079,0.817921
4,0.171750,0.186159,0.169246,0.182511,0.500023,-0.003262,0.334380,156.981173,139.813908,2351,49343,11540,10278,5,Aleatório proporcional à prevalência,0.188552,0.811448


Os resultados obtidos são consistentes com o comportamento esperado de uma baseline sem capacidade discriminativa. As pontuações são geradas aleatoriamente, não utilizando as características das sessões para identificar padrões associados às anomalias.

A Average Precision (AP) apresenta valores próximos da prevalência da classe anómala, embora varie entre os folds devido à aleatoriedade das pontuações e às diferenças na distribuição das classes. O ROC-AUC mantém-se próximo de 0,5 e o MCC próximo de zero, indicando que esta abordagem não consegue distinguir de forma fiável as sessões normais das anómalas.

O recall apresenta valores reduzidos, mostrando que apenas uma parte das sessões anómalas é identificada. A precision também é limitada, pelo que uma proporção considerável das sessões sinalizadas corresponde a falsos positivos. Consequentemente, esta estratégia pode originar falsos alarmes e, simultaneamente, deixar passar muitas sessões anómalas. O Brier score, próximo de 0,33, também reflete a inadequação das pontuações aleatórias como estimativas informadas das probabilidades de anomalia.

Conclui-se que esta baseline não é adequada para a deteção de sessões anómalas. No entanto, constitui uma referência útil para avaliar os modelos seguintes. Espera-se que estes apresentem uma capacidade discriminativa superior, refletida sobretudo em valores mais elevados de AP e ROC-AUC, bem como melhorias no recall, no F2-score e no MCC, mantendo um nível de falsos alarmes operacionalmente aceitável.

### 7.5. Resumir a estabilidade entre folds
Após avaliar cada baseline nos cinco folds temporais, é necessário resumir os resultados para obter uma visão global do seu desempenho e da sua consistência ao longo do tempo.

Para cada métrica, são calculadas a média e o desvio-padrão dos resultados obtidos nos diferentes folds. A média permite avaliar o desempenho médio da baseline, enquanto o desvio-padrão permite perceber quanto esse desempenho varia entre os períodos avaliados. Um desvio-padrão reduzido indica maior consistência dos resultados, embora não implique necessariamente um bom desempenho.

É também identificado o pior fold para cada métrica, permitindo analisar o período em que a baseline apresentou o resultado menos favorável. Para as métricas em que valores mais elevados são melhores (AP, recall, precision, F2-score, ROC-AUC e MCC), considera-se o valor mínimo. Para as métricas em que valores mais baixos são melhores (Brier score, falsos positivos por 1000 sessões e falsos negativos por 1000 sessões), considera-se o valor máximo.

Esta análise facilita a comparação entre as baselines e estabelece uma referência para avaliar os modelos seguintes. Assim, não se considera apenas o desempenho médio, mas também a sua variabilidade e os períodos em que a abordagem apresenta maiores limitações.

In [28]:
def summarize_fold_metrics(results_df, group_name):
    """Resume métricas por baseline, incluindo média, desvio-padrão e pior fold."""
    higher_is_better = {
        "AP (PR-AUC)", "Recall", "Precision", "F2-score", "ROC-AUC", "MCC"
    }
    metric_names = [
        "AP (PR-AUC)", "Recall", "Precision", "F2-score",
        "ROC-AUC", "MCC", "Brier score",
        "FP por 1000 sessões", "FN por 1000 sessões"
    ]

    rows = []
    for metric in metric_names:
        values = results_df[metric].dropna()
        if values.empty:
            mean_value = std_value = worst_value = np.nan
        else:
            mean_value = values.mean()
            std_value = values.std(ddof=1) if len(values) > 1 else 0.0
            worst_value = (
                values.min() if metric in higher_is_better else values.max()
            )
        rows.append({
            "baseline": group_name,
            "métrica": metric,
            "média": mean_value,
            "desvio-padrão": std_value,
            "pior_fold": worst_value,
        })
    return pd.DataFrame(rows)

# Resumo das métricas de cada baseline
summary_parts = [
    ("Classe maioritária",
     summarize_fold_metrics(majority_results_df, "Classe maioritária")),

    ("Aleatório proporcional à prevalência",
     summarize_fold_metrics(
         random_results_df,
         "Aleatório proporcional à prevalência"
     ))
]

# Apresentar uma tabela por baseline
for baseline_name, summary_df in summary_parts:
    print(f"\n{'=' * 60}")
    print(baseline_name)
    print('=' * 60)

    display(summary_df)


Classe maioritária


,baseline,métrica,média,desvio-padrão,pior_fold
0,Classe maioritária,AP (PR-AUC),0.183420,0.019401,0.164000
1,Classe maioritária,Recall,0.000000,0.000000,0.000000
2,Classe maioritária,Precision,0.000000,0.000000,0.000000
3,Classe maioritária,F2-score,0.000000,0.000000,0.000000
4,Classe maioritária,ROC-AUC,0.500000,0.000000,0.500000
5,Classe maioritária,MCC,0.000000,0.000000,0.000000
6,Classe maioritária,Brier score,0.183420,0.019401,0.214441
7,Classe maioritária,FP por 1000 sessões,0.000000,0.000000,0.000000
8,Classe maioritária,FN por 1000 sessões,183.420394,19.400966,214.441180



Aleatório proporcional à prevalência


,baseline,métrica,média,desvio-padrão,pior_fold
0,Aleatório proporcional à prevalência,AP (PR-AUC),0.182691,0.019953,0.161984
1,Aleatório proporcional à prevalência,Recall,0.187682,0.009815,0.178704
2,Aleatório proporcional à prevalência,Precision,0.182876,0.020977,0.161407
3,Aleatório proporcional à prevalência,F2-score,0.186332,0.008914,0.178363
4,Aleatório proporcional à prevalência,ROC-AUC,0.498662,0.004034,0.493942
5,Aleatório proporcional à prevalência,MCC,-0.000689,0.004193,-0.003412
6,Aleatório proporcional à prevalência,Brier score,0.333946,0.001042,0.334864
7,Aleatório proporcional à prevalência,FP por 1000 sessões,153.904124,7.975290,160.858091
8,Aleatório proporcional à prevalência,FN por 1000 sessões,149.017847,16.203074,175.617586


### 7.6. Interpretação e limitações

Os resultados das duas baselines permitem estabelecer referências para avaliar a capacidade de deteção de sessões anómalas e comparar o desempenho de futuras abordagens preditivas.

A baseline da classe maioritária apresenta uma AP (PR-AUC) média de 0,1834 e uma ROC-AUC de 0,5000, indicando ausência de capacidade discriminativa. Como prevê sempre a classe maioritária, não identifica sessões anómalas, apresentando Recall, Precision, F2-score e MCC iguais a zero. Apesar de poder apresentar uma accuracy elevada, esta estratégia não é adequada para a deteção de anomalias, uma vez que falha todas as sessões pertencentes à classe positiva. O número médio de falsos negativos é de 183,42 por cada 1000 sessões. A AP observada constitui uma referência para a avaliação de modelos, devendo ser interpretada considerando a prevalência da classe positiva em cada fold de teste.

A baseline aleatória proporcional à prevalência apresenta uma AP média de 0,1827, próxima da observada na baseline da classe maioritária, e uma ROC-AUC média de 0,4987. O MCC médio é praticamente nulo (-0,0007), o que é consistente com a ausência de uma associação relevante entre as previsões e as classes reais. O Recall médio é de 0,1877 e a Precision média é de 0,1829, valores próximos da prevalência da classe positiva. Esta abordagem regista, em média, 153,90 falsos positivos e 149,02 falsos negativos por cada 1000 sessões. Embora permita identificar algumas sessões anómalas, gera alertas sem uma capacidade discriminativa significativa. A utilização de uma semente aleatória permite reproduzir os resultados, mas a variabilidade entre execuções deve ser considerada na avaliação desta baseline.

A comparação entre as duas abordagens mostra que nenhuma apresenta uma capacidade relevante para distinguir sessões anómalas de sessões normais. A baseline da classe maioritária não deteta qualquer anomalia, enquanto a baseline aleatória identifica algumas sessões anómalas, mas sem demonstrar uma capacidade discriminativa significativa. Estes resultados reforçam a necessidade de desenvolver modelos que utilizem as características disponíveis das sessões para melhorar a qualidade das previsões.

Existem, no entanto, limitações metodológicas importantes. A avaliação deve respeitar a ordem temporal dos dados, garantindo que os modelos utilizam apenas informação disponível no momento da previsão e evitando a utilização de informação futura (data leakage). A avaliação depende também da definição de sessão anómala, da qualidade dos dados e do momento em que a previsão é efetuada.

Por fim, estas baselines constituem referências iniciais e não substituem a avaliação de modelos preditivos treinados com os preditores elegíveis e disponíveis no momento da previsão. Os modelos deverão ser avaliados segundo a mesma estratégia de validação temporal e comparados através das mesmas métricas. O limiar de classificação deverá ser escolhido sem utilizar o conjunto de teste. A escolha da abordagem final deverá considerar não só a capacidade de deteção de anomalias, mas também o equilíbrio entre falsos positivos e falsos negativos, tendo em conta as consequências operacionais de cada tipo de erro.

## 8. Validação temporal e protocolo final

Fronteiras temporais, teste final protegido, folds *expanding-window*, pré-processamento
ajustado dentro de cada fold e agregação dos resultados entre folds. Usa o alvo e os
preditores elegíveis das secções anteriores.

Os folds da secção 7.2 (`TimeSeriesSplit`) dividem por número de linhas e servem as baselines.
Aqui as fronteiras são datas e há purga por `end_time`; este esquema define o protocolo.

In [17]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAL = ["charging_post_id", "location_information", "district_name", "tariff_period"]

# Primeiro dia do teste final; tudo o que começa antes é desenvolvimento.
TEST_START = pd.Timestamp("2021-08-01")
N_FOLDS = 4
FOLD_MONTHS = 3
# Alerta se probabilidade >= THRESHOLD; 1/(1+4) com custo FN/FP = 4 (secção 6), provisório.
THRESHOLD = 0.2

### 8.1 Dados, alvo e preditores elegíveis

In [18]:
y = df["is_abnormal"].astype(int)

eligible = eligible_predictors
excluded = {c for cols in excluded_by_decision.values() for c in cols}
assert not excluded.intersection(eligible), "Preditor excluído presente na lista elegível"

print(f"Sessões: {len(df):,} | período: {start_dt.min():%Y-%m-%d} a {start_dt.max():%Y-%m-%d}")
print(f"Prevalência global: {y.mean():.2%} | preditores elegíveis: {len(eligible)}")

Sessões: 441,077 | período: 2019-12-31 a 2021-12-31
Prevalência global: 18.58% | preditores elegíveis: 37


### 8.2 Fronteiras temporais e teste final protegido

O teste final (a partir de `TEST_START`, cerca de 30% das sessões, tal como o corte 70/30 do
`task1`) não entra em nenhum fold nem em nenhuma decisão de modelação. É avaliado uma única
vez, no fim, com o modelo e o limiar já fixados.

In [19]:
dev_mask = start_dt < TEST_START
test_mask = ~dev_mask
dev_idx = np.flatnonzero(dev_mask)
test_idx = np.flatnonzero(test_mask)

print(f"Desenvolvimento: {dev_mask.sum():,} sessões ({dev_mask.mean():.1%}), "
      f"{start_dt[dev_mask].min():%Y-%m-%d} a {start_dt[dev_mask].max():%Y-%m-%d}")
print(f"Teste final protegido: {test_mask.sum():,} sessões ({test_mask.mean():.1%}), "
      f"{start_dt[test_mask].min():%Y-%m-%d} a {start_dt[test_mask].max():%Y-%m-%d}")

Desenvolvimento: 309,930 sessões (70.3%), 2019-12-31 a 2021-07-31
Teste final protegido: 131,147 sessões (29.7%), 2021-08-01 a 2021-12-31


### 8.3 Folds expanding-window com purga

Cada fold treina com tudo o que terminou antes do início da validação e valida nos três meses
seguintes. A purga retira do treino as sessões que começaram antes da fronteira mas cujo
resultado ainda não era conhecido nessa data. As janelas de validação são contíguas e a
última termina em `TEST_START`; o primeiro fold tem menos treino (cerca de 7 meses).

In [20]:
val_starts = [TEST_START - pd.DateOffset(months=FOLD_MONTHS * (N_FOLDS - k)) for k in range(N_FOLDS)]
val_ends = val_starts[1:] + [TEST_START]

folds, fold_rows = [], []
for k, (v_start, v_end) in enumerate(zip(val_starts, val_ends), start=1):
    in_val = ((start_dt >= v_start) & (start_dt < v_end)).to_numpy()
    started_before = (start_dt < v_start).to_numpy()
    in_train = started_before & (end_dt < v_start).to_numpy()
    train_idx, val_idx = np.flatnonzero(in_train), np.flatnonzero(in_val)
    folds.append((train_idx, val_idx))
    fold_rows.append({
        "fold": k,
        "treino_inicio": start_dt.iloc[train_idx].min().date(),
        "treino_fim": v_start.date(),
        "validacao_fim": v_end.date(),
        "n_treino": len(train_idx),
        "n_validacao": len(val_idx),
        "purgadas": int((started_before & ~in_train).sum()),
        "prevalencia_treino": y.iloc[train_idx].mean(),
        "prevalencia_validacao": y.iloc[val_idx].mean(),
    })

fold_table = pd.DataFrame(fold_rows)
pd.set_option("display.width", 200)
fold_table.round(4)

,fold,treino_inicio,treino_fim,validacao_fim,n_treino,n_validacao,purgadas,prevalencia_treino,prevalencia_validacao
0,1,2019-12-31,2020-08-01,2020-11-01,79695,66045,36,0.1966,0.1868
1,2,2019-12-31,2020-11-01,2021-02-01,145748,57027,28,0.1921,0.1679
2,3,2019-12-31,2021-02-01,2021-05-01,202793,42189,10,0.1853,0.1532
3,4,2019-12-31,2021-05-01,2021-08-01,244950,64938,42,0.1798,0.1943


### 8.4 Verificações de consistência do esquema

In [21]:
for k, (tr, va) in enumerate(folds, start=1):
    assert np.intersect1d(tr, va).size == 0, f"fold {k}: treino e validação sobrepõem-se"
    assert end_dt.iloc[tr].max() < start_dt.iloc[va].min(), f"fold {k}: treino com informação posterior"
    assert np.intersect1d(np.concatenate([tr, va]), test_idx).size == 0, f"fold {k}: toca no teste final"
    assert len(tr) > 0 and len(va) > 0
assert start_dt[dev_mask].max() < start_dt[test_mask].min()
assert all(folds[i][1].max() < folds[i + 1][1].min() for i in range(N_FOLDS - 1))
assert val_ends[-1] == TEST_START
assert len(dev_idx) + len(test_idx) == len(df)
print("Esquema consistente: sem sobreposição, sem informação futura, teste final intocado.")

Esquema consistente: sem sobreposição, sem informação futura, teste final intocado.


### 8.5 Pré-processamento dentro de cada fold

As transformações sem estado (hora e dia da semana cíclicos) são calculadas uma vez.
Imputação, escala e `OneHotEncoder` fazem parte do `Pipeline`, que é clonado e ajustado só com
o treino de cada fold, para que nada da validação influencie as estatísticas aprendidas.

In [22]:
def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    X = frame[eligible].copy()
    hour = start_dt.dt.hour + start_dt.dt.minute / 60
    X["hour_sin"], X["hour_cos"] = np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24)
    X["weekday_sin"] = np.sin(2 * np.pi * start_dt.dt.weekday / 7)
    X["weekday_cos"] = np.cos(2 * np.pi * start_dt.dt.weekday / 7)
    X = X.drop(columns="start_time")
    X[CATEGORICAL] = X[CATEGORICAL].astype(str)
    numeric = [c for c in X.columns if c not in CATEGORICAL]
    X[numeric] = X[numeric].astype(float)
    return X


X = build_features(df)
numeric_cols = [c for c in X.columns if c not in CATEGORICAL]

preprocessing = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CATEGORICAL),
])
# Modelo de referência só para exercitar o esquema; o modelo final é escolhido depois.
reference_model = Pipeline([
    ("prep", preprocessing),
    ("clf", LogisticRegression(max_iter=1000)),
])

### 8.6 Resultados por fold e baselines

Dois baselines por fold: a prevalência (PR-AUC de um classificador sem poder discriminativo)
e a taxa histórica de anomalias do posto nos últimos 30 dias, sem treino.

In [23]:
def fold_metrics(y_true, score, threshold=THRESHOLD):
    alert = score >= threshold
    tp = int((alert & (y_true == 1)).sum())
    fp = int((alert & (y_true == 0)).sum())
    fn = int((~alert & (y_true == 1)).sum())
    n = len(y_true)
    return {
        "pr_auc": average_precision_score(y_true, score),
        "roc_auc": roc_auc_score(y_true, score),
        "brier": brier_score_loss(y_true, score),
        "precision": tp / (tp + fp) if tp + fp else np.nan,
        "recall": tp / (tp + fn) if tp + fn else np.nan,
        "alertas_por_1000": 1000 * alert.sum() / n,
        "fp_por_1000": 1000 * fp / n,
        "fn_por_1000": 1000 * fn / n,
    }


history_score = df["post_previous_abnormal_rate_30d"].fillna(0).to_numpy()
result_rows, pooled_y, pooled_p = [], [], []
for k, (tr, va) in enumerate(folds, start=1):
    model = clone(reference_model).fit(X.iloc[tr], y.iloc[tr])
    p = model.predict_proba(X.iloc[va])[:, 1]
    y_va = y.iloc[va].to_numpy()
    result_rows.append({
        "fold": k, "n_val": len(va), **fold_metrics(y_va, p),
        "baseline_prevalencia_pr_auc": y_va.mean(),
        "baseline_historico_pr_auc": average_precision_score(y_va, history_score[va]),
    })
    pooled_y.append(y_va)
    pooled_p.append(p)

fold_results = pd.DataFrame(result_rows)
fold_results.round(4)

,fold,n_val,pr_auc,roc_auc,brier,precision,recall,alertas_por_1000,fp_por_1000,fn_por_1000,baseline_prevalencia_pr_auc,baseline_historico_pr_auc
0,1,66045,0.6159,0.8273,0.1070,0.4690,0.6656,265.0920,140.7525,62.4726,0.1868,0.4604
1,2,57027,0.5777,0.8153,0.1023,0.4339,0.6011,232.6267,131.6920,66.9858,0.1679,0.4387
2,3,42189,0.5010,0.8069,0.1011,0.3988,0.5867,225.3668,135.4856,63.3103,0.1532,0.3617
3,4,64938,0.6353,0.8342,0.1079,0.4869,0.6671,266.2848,136.6380,64.6925,0.1943,0.4688


### 8.7 Agregação entre folds

Métrica principal: média simples (cada período conta o mesmo) com desvio-padrão. A média
ponderada pelo tamanho da validação serve de verificação de sensibilidade, o pior fold mostra o
risco e a tendência (declive por fold) indica melhoria ou degradação ao longo do tempo. Com
apenas 4 folds, a tendência é indicativa e não um teste estatístico.

In [24]:
def aggregate(res: pd.DataFrame, metric: str, higher_is_better: bool = True) -> dict:
    v, w = res[metric], res["n_val"]
    return {
        "media": v.mean(),
        "desvio_padrao": v.std(ddof=1),
        "media_ponderada": np.average(v, weights=w),
        "pior_fold": v.min() if higher_is_better else v.max(),
        "tendencia_por_fold": np.polyfit(res["fold"], v, 1)[0],
    }


fold_summary = pd.DataFrame({
    "pr_auc": aggregate(fold_results, "pr_auc"),
    "roc_auc": aggregate(fold_results, "roc_auc"),
    "brier": aggregate(fold_results, "brier", higher_is_better=False),
    "recall": aggregate(fold_results, "recall"),
    "precision": aggregate(fold_results, "precision"),
    "alertas_por_1000": aggregate(fold_results, "alertas_por_1000"),
    "baseline_prevalencia_pr_auc": aggregate(fold_results, "baseline_prevalencia_pr_auc"),
    "baseline_historico_pr_auc": aggregate(fold_results, "baseline_historico_pr_auc"),
}).T
fold_summary.round(4)

,media,desvio_padrao,media_ponderada,pior_fold,tendencia_por_fold
pr_auc,0.5825,0.0594,0.5909,0.5010,-0.0019
roc_auc,0.8209,0.0122,0.8225,0.8069,0.0012
brier,0.1046,0.0034,0.1050,0.1079,0.0001
recall,0.6301,0.0422,0.6356,0.5867,-0.0010
precision,0.4472,0.0390,0.4525,0.3988,0.0018
alertas_por_1000,247.3425,21.3958,250.1053,225.3668,-0.3682
baseline_prevalencia_pr_auc,0.1756,0.0186,0.1781,0.1532,0.0008
baseline_historico_pr_auc,0.4324,0.0488,0.4393,0.3617,-0.0052


In [25]:
pooled_y, pooled_p = np.concatenate(pooled_y), np.concatenate(pooled_p)
print(f"PR-AUC pooled (só informativa; mistura modelos de folds diferentes): "
      f"{average_precision_score(pooled_y, pooled_p):.4f}")

PR-AUC pooled (só informativa; mistura modelos de folds diferentes): 0.5967


### 8.7b Seleção de preditores por grupos

Os 37 preditores elegíveis da secção 4 foram aceites por disponibilidade no início da sessão e ausência de fuga. Aqui verifica-se que grupos de variáveis contribuem de facto para o desempenho, usando só os folds de validação e o modelo de referência. O teste final não é usado.

**Método.** As variáveis são agrupadas por significado.
1. **Ablação:** mede-se a variação da PR-AUC em cada fold ao retirar cada grupo, mantendo os restantes.
2. **Eliminação sequencial:** em cada passo retira-se o grupo cuja remoção prejudica menos. O processo pára quando, face ao modelo completo, a PR-AUC média cai mais de 0,001 ou algum fold cai mais de 0,003. Estas tolerâncias são uma escolha do protocolo, do tamanho do ruído entre variantes quase equivalentes.

O pré-processamento continua dentro do `Pipeline`, ajustado só com o treino de cada fold.

In [26]:
import warnings

from joblib import Parallel, delayed

warnings.filterwarnings("ignore", message="A worker stopped")

PREDICTOR_GROUPS = {
    "posto_historico": [c for c in X.columns if c.startswith("post_previous_")],
    "posto_recencia": [c for c in X.columns if c.startswith(("post_hours_since", "post_no_previous"))],
    "utilizador_historico": [c for c in X.columns if c.startswith("user_previous_")],
    "utilizador_recencia": [
        c for c in X.columns
        if c.startswith(("user_days_since", "user_no_previous")) or c == "is_new_user"
    ],
    "utilizador_sessoes_ativas": ["user_active_sessions_at_start"],
    "meteorologia": ["temperature", "relative_humidity", "precipitation_mm"],
    "tarifa": ["electricity_price", "tariff_period"],
    "local": ["charging_post_id", "location_information", "district_name"],
    "hora_semana": ["hour_sin", "hour_cos", "weekday_sin", "weekday_cos"],
}
assert sorted(sum(PREDICTOR_GROUPS.values(), [])) == sorted(X.columns), "Há colunas sem grupo"

SELECTION_TOL_MEAN, SELECTION_TOL_FOLD = 0.001, 0.003


def build_model(columns):
    num = [c for c in columns if c not in CATEGORICAL]
    cat = [c for c in columns if c in CATEGORICAL]
    parts = []
    if num:
        parts.append(("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num))
    if cat:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cat))
    return Pipeline([("prep", ColumnTransformer(parts)), ("clf", LogisticRegression(max_iter=1000))])


def fit_eval(columns, k):
    tr, va = folds[k]
    model = build_model(columns).fit(X.iloc[tr][columns], y.iloc[tr])
    p = model.predict_proba(X.iloc[va][columns])[:, 1]
    return fold_metrics(y.iloc[va].to_numpy(), p)


def evaluate_group_sets(group_sets):
    """Métricas por fold (uma tabela por conjunto de grupos)."""
    column_sets = [[c for c in X.columns if any(c in PREDICTOR_GROUPS[g] for g in gs)] for gs in group_sets]
    jobs = [(i, k) for i in range(len(column_sets)) for k in range(N_FOLDS)]
    out = Parallel(n_jobs=4)(delayed(fit_eval)(column_sets[i], k) for i, k in jobs)
    return [pd.DataFrame(out[i * N_FOLDS:(i + 1) * N_FOLDS]) for i in range(len(group_sets))]

**Ablação.** Cada linha retira um grupo. `delta` é a PR-AUC sem o grupo menos a do modelo completo (valores negativos significam que o grupo ajuda).

In [27]:
all_groups = list(PREDICTOR_GROUPS)
full_pr = evaluate_group_sets([all_groups])[0]["pr_auc"].to_numpy()
without = evaluate_group_sets([[g for g in all_groups if g != d] for d in all_groups])
deltas = [w["pr_auc"].to_numpy() - full_pr for w in without]

ablation = pd.DataFrame({
    "n_colunas": [len(PREDICTOR_GROUPS[g]) for g in all_groups],
    "pr_auc_sem_grupo": [w["pr_auc"].mean() for w in without],
    "delta_medio": [d.mean() for d in deltas],
    "delta_pior_fold": [d.min() for d in deltas],
    "folds_que_melhoram": [int((d > 0).sum()) for d in deltas],
}, index=all_groups).sort_values("delta_medio")
print(f"PR-AUC média do modelo completo ({X.shape[1]} colunas): {full_pr.mean():.4f}")
ablation.round(4)

PR-AUC média do modelo completo (40 colunas): 0.5825


,n_colunas,pr_auc_sem_grupo,delta_medio,delta_pior_fold,folds_que_melhoram
posto_historico,9,0.4183,-0.1641,-0.2217,0
utilizador_historico,9,0.5615,-0.0210,-0.0274,0
utilizador_recencia,5,0.5819,-0.0006,-0.0017,1
local,3,0.5820,-0.0005,-0.0023,2
posto_recencia,4,0.5822,-0.0003,-0.0009,1
hora_semana,4,0.5823,-0.0002,-0.0007,1
tarifa,2,0.5823,-0.0002,-0.0004,1
meteorologia,3,0.5825,-0.0000,-0.0007,3
utilizador_sessoes_ativas,1,0.5828,0.0003,-0.0003,2


**Eliminação sequencial.** Em cada passo testa-se retirar cada grupo que ainda resta e escolhe-se o que deixa a PR-AUC média mais alta. O passo é aceite se respeitar as duas tolerâncias face ao modelo **completo**, e não face ao passo anterior, para que pequenas perdas não se acumulem.

In [28]:
current = list(all_groups)
steps = []
while len(current) > 1:
    candidates = [[g for g in current if g != d] for d in current]
    scored = evaluate_group_sets(candidates)
    pr = np.array([s["pr_auc"].to_numpy() for s in scored])
    best = int(np.argmax(pr.mean(axis=1)))
    delta = pr[best] - full_pr
    accepted = bool(delta.mean() >= -SELECTION_TOL_MEAN and delta.min() >= -SELECTION_TOL_FOLD)
    steps.append({
        "passo": len(steps) + 1, "grupo_retirado": current[best], "pr_auc_medio": pr[best].mean(),
        "delta_medio": delta.mean(), "delta_pior_fold": delta.min(), "aceite": accepted,
    })
    if not accepted:
        break
    current = candidates[best]

selected_groups = current
selected_predictors = [c for c in X.columns if any(c in PREDICTOR_GROUPS[g] for g in selected_groups)]
display(pd.DataFrame(steps).round(4))
print(f"\nGrupos mantidos: {selected_groups}")
print(f"Preditores selecionados ({len(selected_predictors)} de {X.shape[1]}):")
print(selected_predictors)

,passo,grupo_retirado,pr_auc_medio,delta_medio,delta_pior_fold,aceite
0,1,utilizador_sessoes_ativas,0.5828,0.0003,-0.0003,True
1,2,local,0.5828,0.0003,-0.0024,True
2,3,meteorologia,0.5831,0.0006,-0.0024,True
3,4,tarifa,0.5831,0.0006,-0.0021,True
4,5,hora_semana,0.5827,0.0002,-0.0019,True
5,6,posto_recencia,0.5823,-0.0002,-0.0027,True
6,7,utilizador_recencia,0.5820,-0.0005,-0.0025,True
7,8,utilizador_historico,0.5454,-0.0371,-0.0486,False



Grupos mantidos: ['posto_historico', 'utilizador_historico']
Preditores selecionados (18 de 40):
['post_previous_sessions_1d', 'post_previous_abnormal_1d', 'post_previous_abnormal_rate_1d', 'post_previous_sessions_7d', 'post_previous_abnormal_7d', 'post_previous_abnormal_rate_7d', 'post_previous_sessions_30d', 'post_previous_abnormal_30d', 'post_previous_abnormal_rate_30d', 'user_previous_sessions_30d', 'user_previous_abnormal_30d', 'user_previous_abnormal_rate_30d', 'user_previous_sessions_180d', 'user_previous_abnormal_180d', 'user_previous_abnormal_rate_180d', 'user_previous_sessions_730d', 'user_previous_abnormal_730d', 'user_previous_abnormal_rate_730d']


**Modelo completo vs selecionado.** Comparação nos mesmos folds.

In [29]:
full_res, sel_res = evaluate_group_sets([all_groups, selected_groups])
for res in (full_res, sel_res):
    res["fold"] = range(1, N_FOLDS + 1)
    res["n_val"] = [len(v) for _, v in folds]

selection_metrics = ["pr_auc", "roc_auc", "recall", "precision", "alertas_por_1000"]
comparison = []
for name, res in ((f"completo ({X.shape[1]} colunas)", full_res), (f"selecionado ({len(selected_predictors)} colunas)", sel_res)):
    for m in selection_metrics:
        a = aggregate(res, m)
        comparison.append({"modelo": name, "métrica": m, "media": a["media"], "desvio_padrao": a["desvio_padrao"]})
pd.DataFrame(comparison).pivot(index="métrica", columns="modelo", values=["media", "desvio_padrao"]).loc[selection_metrics].round(4)

media                                  desvio_padrao                         
modelo           completo (40 colunas) selecionado (18 colunas) completo (40 colunas) selecionado (18 colunas)
métrica                                                                                                       
pr_auc                          0.5825                   0.5820                0.0594                   0.0595
roc_auc                         0.8209                   0.8190                0.0122                   0.0135
recall                          0.6301                   0.6143                0.0422                   0.0541
precision                       0.4472                   0.4575                0.0390                   0.0337
alertas_por_1000              247.3425                 235.8138               21.3958                  28.6733

**Leitura dos resultados.**
- Só dois grupos contribuem de forma mensurável: o histórico do posto (sem ele, a PR-AUC média desce de 0,583 para 0,418) e o histórico do utilizador (desce 0,021). Os outros sete grupos alteram a PR-AUC em menos de 0,001, em média.
- A eliminação sequencial retira sete grupos e pára no histórico do utilizador (perda média de 0,037 e de 0,049 no pior fold).
- O conjunto selecionado tem 18 colunas: histórico do posto a 1, 7 e 30 dias, e histórico do utilizador a 30, 180 e 730 dias. Mantém a PR-AUC média (0,582 contra 0,583) e a ROC-AUC (0,819 contra 0,821). Com o limiar 0,2, o recall é 0,614 (0,630 no completo), a precision é 0,458 (0,447) e os alertas passam de 247 para 236 por 1000 sessões.
- Meteorologia, tarifa, hora e local não têm efeito mensurável neste modelo. Isso não prova que não se relacionem com as anomalias: o histórico pode já conter essa informação, ou o efeito ser pequeno.

**Limitações.** A seleção foi feita com a regressão logística; modelos não lineares podem aproveitar variáveis (por exemplo, interações entre hora e posto) que um modelo linear não usa, por isso deve ser repetida com o modelo final. A escolha usa os mesmos 4 folds em que se reporta o resultado, o que a torna ligeiramente otimista; o teste final dá a estimativa sem esse viés.

### 8.8 Protocolo final de avaliação

**Alvo e dados.** `is_abnormal` = `end_cause` fora de {*Charging ends normally*, *User stops charging*} (18,58% das sessões). Preditores: os 37 elegíveis da secção 4 (inclui as 26 colunas de histórico); excluídos `end_cause`, `is_abnormal`, as colunas de fuga, `userid` e `order_creation_time`. A meteorologia é usada como previsão disponível no início da sessão, como declara o enunciado.

**Partição temporal.** Desenvolvimento: 2019-12-31 a 2021-07-31 (309 930 sessões, 70,3%). Teste final: 2021-08-01 a 2021-12-31 (131 147 sessões, 29,7%), protegido e avaliado uma só vez com modelo e limiar já fixados.

**Validação.** *Expanding-window* com 4 folds de 3 meses (validações: 2020-08 a 10, 2020-11 a 2021-01, 2021-02 a 04, 2021-05 a 07). O treino de cada fold usa só sessões que terminaram antes da validação (purga de 10 a 42 sessões por fold). O pré-processamento (imputação, escala, *one-hot*) é ajustado dentro do `Pipeline` com o treino de cada fold. O teste final não participa em nenhuma escolha.

**Métricas.** Principal: PR-AUC (classe positiva rara). Secundárias: ROC-AUC, Brier, precision e recall no limiar. Operacionais por 1000 sessões: alertas, falsos positivos e falsos negativos.

**Limiar.** O modelo devolve uma probabilidade de anomalia e a sessão gera alerta se esta for maior ou igual ao limiar. Com o custo de um falso negativo igual a 4 vezes o de um falso positivo (hipótese da secção 6, a validar com a operação), o limiar que minimiza o custo esperado é $1/(1+4)=0{,}2$, abaixo dos 0,5 habituais. Baixá-lo dá mais alertas e mais recall, com menos precision. A PR-AUC e a ROC-AUC não dependem do limiar; recall, precision e alertas dependem. Aqui é fixo e provisório; na modelação será escolhido só com os folds de validação, nunca com o teste final.

**Baselines.** Prevalência (PR-AUC igual à prevalência do fold, média 0,176) e taxa histórica de anomalias do posto nos últimos 30 dias (PR-AUC média 0,432). Qualquer modelo tem de superar o segundo, não só o primeiro.

**Seleção de preditores.** Feita por grupos de variáveis (secção 8.7b), com os mesmos folds e só com o treino de cada fold; o teste final não é usado. Uma remoção é aceite se a PR-AUC média não cair mais de 0,001 nem nenhum fold mais de 0,003 face ao modelo completo. Com o modelo de referência ficam 18 preditores: histórico do posto (1, 7 e 30 dias) e histórico do utilizador (30, 180 e 730 dias). A seleção repete-se com o modelo final.

**Agregação.** Média simples dos folds com desvio-padrão e pior fold; média ponderada pelo tamanho como verificação de sensibilidade; declive por fold para detetar degradação. A agregação *pooled* só serve para escolher o limiar, porque mistura modelos de folds diferentes.

**Resultado do modelo de referência** (regressão logística, limiar provisório 0,2): PR-AUC 0,583 ± 0,059 (pior fold 0,501), ROC-AUC 0,821 ± 0,012, recall 0,630, precision 0,447, cerca de 247 alertas por 1000 sessões. A média ponderada (0,591) é próxima da simples, e não há tendência clara de degradação (declive de PR-AUC de -0,002 por fold). A variação entre folds acompanha a prevalência da validação (0,153 a 0,194).

**Limitações.** Com 4 folds, o desvio-padrão e a tendência são indicativos. O limiar 0,2 gera um nível de alertas alto e é provisório até à modelação.

**Verificação de consistência.** Alvo, preditores e exclusões coincidem com a tabela de decisões da secção 4. Os folds não se sobrepõem, não usam informação posterior à fronteira e não tocam no teste final (secção 8.4). A partição 70/30 coincide com a do `task1`.